# Blackbody radiation: Planck vs. Rayleigh–Jeans

**Professor Nano Labs · Quantum foundations**

**Try it live in your browser:** [Blackbody Radiation lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/blackbody-radiation/)  
**Watch first:** [The light wave and particle debate, from Newton to Einstein](https://www.youtube.com/watch?v=jpu36CYydkc)  
**Watch later:** [Fermi Dirac vs Bose Einstein vs Classical Maxwell Boltzmann distributions](https://www.youtube.com/watch?v=e9TQs-EDiUs) · [Deriving the Bose Einstein distribution function](https://www.youtube.com/watch?v=BLqMA8YWkDY)

Light inside a hot cavity is a set of standing-wave modes, and the number of modes per unit wavelength grows as $1/\lambda^4$. Classical physics gives every mode the same average energy $k_BT$, which yields the **Rayleigh–Jeans law**. It agrees with experiment at long wavelengths but predicts infinite power at short ones: the *ultraviolet catastrophe*. **Planck's law** fixes this by letting each mode hold energy only in quanta $hc/\lambda$:

$$B_\lambda^{\rm RJ}(T)=\frac{2ck_BT}{\lambda^4},\qquad
B_\lambda(T)=\frac{2hc^2}{\lambda^5}\,\frac{1}{e^{hc/\lambda k_BT}-1}.$$

$B_\lambda$ is the spectral radiance (W m$^{-2}$ sr$^{-1}$ per metre of wavelength). The energy density inside the cavity is $u_\lambda = 4\pi B_\lambda/c$, which gives the $8\pi$ forms used below.

**A note on the code.** The denominator is $e^x-1$ with $x=hc/\lambda k_BT$. NumPy's `np.expm1(x)` computes exactly $e^x-1$, accurately even when $x$ is small, so it is used on its own. Writing `np.expm1(x) - 1` would give $e^x-2$, which inflates the long-wavelength tail and turns negative beyond about 5.2 µm at 4000 K.

In [ ]:
# To run this code numpy and matplotlib are needed. To install them use
# pip install numpy matplotlib
# In the case this doesn't work, try
# pip3 install numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Constants (exact SI values since 2019)
c = 299792458.0        # Speed of light in vacuum (m/s)
h = 6.62607015e-34     # Planck's constant (J*s)
k_B = 1.380649e-23     # Boltzmann constant (J/K)
T = 4000               # Temperature in Kelvin (K)

# Wavelength range (in meters)
l_min = 1e-9
l_max = 3.0e-6
wavelength = np.linspace(l_min, l_max, 1000)  # from 1 nm to 3 µm

# Rayleigh-Jeans law (spectral radiance)
def rayleigh_jeans_radiance(wavelength, T):
    return (2 * c * k_B * T) / (wavelength ** 4)

# Rayleigh-Jeans law (spectral energy density)
def rayleigh_jeans_density(wavelength, T):
    return (8 * np.pi * k_B * T) / (wavelength ** 4)

# Planck's law (spectral radiance).
# np.expm1(x) is already e^x - 1. At very short wavelengths e^x overflows to
# infinity and the radiance correctly becomes 0; errstate silences that warning.
def planck_law_radiance(wavelength, T):
    exponent = h * c / (wavelength * k_B * T)
    with np.errstate(over='ignore'):
        return (2 * h * c**2) / (wavelength ** 5 * np.expm1(exponent))

# Planck's law (spectral energy density)
def planck_law_density(wavelength, T):
    exponent = h * c / (wavelength * k_B * T)
    with np.errstate(over='ignore'):
        return (8 * np.pi * h * c) / (wavelength ** 5 * np.expm1(exponent))

# Spectral radiance
rj_radiance = rayleigh_jeans_radiance(wavelength, T)
planck_radiance = planck_law_radiance(wavelength, T)

# Spectral energy density
rj_density = rayleigh_jeans_density(wavelength, T)
planck_density = planck_law_density(wavelength, T)

In [ ]:
# Plotting: spectral radiance
plt.figure(figsize=(10, 6))
plt.plot(wavelength * 1e9, rj_radiance, label='Rayleigh-Jeans Law', linestyle='--')
plt.plot(wavelength * 1e9, planck_radiance, label="Planck's Law", color='red')
plt.ylim(0, 15e12)
plt.xlim(l_min * 1e9, l_max * 1e9)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Spectral radiance (W m$^{-2}$ sr$^{-1}$ m$^{-1}$)')
plt.title(f'Spectral radiance T = {T} K')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Plotting: spectral energy density
plt.figure(figsize=(10, 6))
plt.plot(wavelength * 1e9, rj_density, label='Rayleigh-Jeans Law', linestyle='--')
plt.plot(wavelength * 1e9, planck_density, label="Planck's Law", color='red')
plt.ylim(0, 2e5)
plt.xlim(l_min * 1e9, l_max * 1e9)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Spectral energy density (J m$^{-3}$ m$^{-1}$)')
plt.title(f'Spectral density T = {T} K')
plt.legend()
plt.grid(True)
plt.show()

## Checks: Wien's law, Stefan–Boltzmann and the classical limit

Three independent results test the curves above:

* **Wien's displacement law.** The peak sits at $\lambda_{\max}=b/T$, with $b = 2.897771955\times10^{-3}$ m K.
* **Stefan–Boltzmann law.** Integrating $B_\lambda$ over all wavelengths gives $\sigma T^4/\pi$, with $\sigma = 2\pi^5k_B^4/(15h^3c^2)$.
* **Classical limit.** At long wavelengths ($hc/\lambda \ll k_BT$) Planck's law approaches Rayleigh–Jeans.

In [ ]:
trapezoid = getattr(np, "trapezoid", None) or np.trapz   # NumPy 2 renamed trapz

# Wien's displacement law
b_wien = 2.897771955e-3                       # m K
fine = np.linspace(1e-9, 100e-6, 2_000_001)   # 1 nm to 100 µm in 50 pm steps
B = planck_law_radiance(fine, T)
print(f"Peak of the curve: {fine[np.argmax(B)] * 1e9:.1f} nm;  Wien's law b/T: {b_wien / T * 1e9:.1f} nm")

# Stefan-Boltzmann law (the range above holds all but a few millionths of the total at 4000 K)
sigma = 2 * np.pi**5 * k_B**4 / (15 * h**3 * c**2)
total = trapezoid(B, fine)
print(f"Integral of B: {total:.6e} W m^-2 sr^-1;  sigma T^4 / pi: {sigma * T**4 / np.pi:.6e}  (ratio {total / (sigma * T**4 / np.pi):.6f})")

# Classical limit: the ratio Rayleigh-Jeans / Planck approaches 1 at long wavelengths
for lam in (0.5e-6, 3e-6, 30e-6, 300e-6):
    ratio = rayleigh_jeans_radiance(lam, T) / planck_law_radiance(lam, T)
    print(f"At {lam * 1e6:6.1f} µm, Rayleigh-Jeans / Planck = {ratio:10.4f}")

## Things to try

Each idea matches a "Predict first" question in the [browser lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/blackbody-radiation/).

1. **The Sun.** Set `T = 5772`. Where does the peak move, and by what factor does $\sigma T^4$ grow compared with 4000 K?
2. **Your own glow.** Set `T = 310` and `l_max = 50e-6`, and raise the plot limits. Where is the peak?
3. **Doubling the temperature.** Plot `T = 3000` and `T = 6000` on the same axes. Check that the peak height grows as $T^5$ and the area as $T^4$.
4. **Where the classical picture fails.** Plot `planck_law_radiance / rayleigh_jeans_radiance`, which equals $x/(e^x-1)$, the average energy per mode in units of $k_BT$. At what wavelength does it fall to one half?